# NeoOLAF native EventStoryLine layer ablation — one document v1.2

This v1.2 notebook uses the same **OWL-Time** seed ontology as the RAGTree EventStoryLine experiments, while retaining the controlled normalized task labels `PRECONDITION` and `FALLING_ACTION` for mapping and evaluation.

It runs one complete native NeoOLAF Layer 0--12 pipeline, saves all artifacts, and loads gold annotations only after execution.


In [1]:
from __future__ import annotations

import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(env_name: str, candidates: list[Path]) -> Path:
    raw = os.environ.get(env_name, "").strip().strip('"').strip("'")
    if raw:
        path = Path(raw).expanduser().resolve()
        if path.is_file():
            return path
        raise FileNotFoundError(f"{env_name} points to a missing file: {path}")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {env_name}. Tried:\n"
        + "\n".join(str(Path(x).expanduser().resolve()) for x in candidates)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from eventstoryline_native_ablation_v1_2 import (
    RELATION_IDS,
    analyze_run,
    gold_event_index,
    indexed_token_table,
    load_layer_states,
    project_event_label,
    read_json,
    read_jsonl,
    run_native_pipeline,
    seed_ontology_summary,
)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /home/galencarmedeiro/git/postdoc/NeoOLAF


/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [2]:
INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_input_v1.jsonl"
GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_gold_v1.jsonl"
SMOKE5_INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_input_v1.jsonl"
SMOKE5_GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_gold_v1.jsonl"

# Same seed ontology used by the RAGTree EventStoryLine experiments.
# Override explicitly with EVENTSTORYLINE_ONTOLOGY_PATH when needed.
ONTOLOGY_PATH = first_existing_path(
    "EVENTSTORYLINE_ONTOLOGY_PATH",
    [
        PROJECT_ROOT.parent / "ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT.parent / "RAGTree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../RAGTree/data/ontology/OWLTime/time.ttl",
    ],
)

# Controlled normalized benchmark relation schema. This is not the seed ontology.
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/eventstoryline_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/eventstoryline_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_profile_native_ablation_v1_2.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_eventstoryline_native_ablation_v1_2.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_task_guidance_v1_2.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/eventstoryline_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "document_1_10ecbplus_v1_2_owltime_validated_events"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Relation-instance decisions are independent and run concurrently in Layer 2.
WORKERS = 16
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

print("Input:", INPUT_JSONL)
print("Gold:", GOLD_JSONL)
print("OWL-Time seed ontology:", ONTOLOGY_PATH)
print("Controlled relation catalog:", RELATION_CATALOG)
print("Profile:", PROFILE_PATH)
print("Guidance:", GUIDANCE_PATH)
print("Task guidance:", TASK_GUIDANCE_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))


Input: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/data/eventstoryline_one_input_v1.jsonl
Gold: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/data/eventstoryline_one_gold_v1.jsonl
OWL-Time seed ontology: /home/galencarmedeiro/git/postdoc/ragtree/data/ontology/OWLTime/time.ttl
Controlled relation catalog: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/ontology/eventstoryline_relation_catalog.json
Profile: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/eventstoryline_profile_native_ablation_v1_2.json
Guidance: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/guidance_eventstoryline_native_ablation_v1_2.json
Task guidance: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/eventstoryline_task_guidance_v1_2.json
Run dir: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v

## 2. Preflight: anti-leakage, OWL-Time seed ontology, five-document files and event identity


In [3]:
required = [
    INPUT_JSONL, GOLD_JSONL, SMOKE5_INPUT_JSONL, SMOKE5_GOLD_JSONL,
    ONTOLOGY_PATH, RELATION_CATALOG, RELATION_ALIASES,
    PROFILE_PATH, GUIDANCE_PATH, TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
smoke_input_rows = read_jsonl(SMOKE5_INPUT_JSONL)
smoke_gold_rows = read_jsonl(SMOKE5_GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert len(smoke_input_rows) == 5 and len(smoke_gold_rows) == 5
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert all("entities" not in row and "relations" not in row for row in smoke_input_rows)
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]
assert [x["document_id"] for x in smoke_input_rows] == [x["document_id"] for x in smoke_gold_rows]

profile = read_json(PROFILE_PATH)
task = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)
gold = gold_rows[0]
seed_summary = seed_ontology_summary(ONTOLOGY_PATH)

print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Sentences:", len(input_rows[0]["sentences"]))
print("Source tokens:", sum(len(x) for x in input_rows[0]["tokens"]))
print("Gold events (not exposed to pipeline):", len(gold["entities"]))
print("Gold evaluated relations:", sum(len(v) for k, v in gold["relations"].items() if k in RELATION_IDS))
print("Ignored null pairs:", len(gold["relations"].get("null", [])))
print("OWL-Time classes loaded:", seed_summary["class_count"])
print("OWL-Time properties loaded:", seed_summary["property_count"])
print("Controlled task relations:", catalog["property_count"])
print("Relation IDs:", task["allowed_relation_ids"])
print("Layer 2 workers:", profile["layers"]["layer02_candidate_enrichment"]["max_concurrency"])
print("Mention-free relation schemas injected:", len(profile["relations"]["allowed"]))
print("Five-document input IDs:", [row["document_id"] for row in smoke_input_rows])

assert seed_summary["class_count"] > 0 or seed_summary["property_count"] > 0
assert catalog["property_count"] == 2
assert set(task["allowed_relation_ids"]) == set(RELATION_IDS)
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_eventstoryline_extraction"] is False
assert profile["anti_cheating"]["source_event_anchoring"] is False
assert profile["anti_cheating"]["gold_pair_hints"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert profile["benchmark_projection"]["same_seed_ontology_as_ragtree"] is True

display(Markdown("### Indexed source table used by Layer 1"))
print(indexed_token_table(input_rows[0]["sentences"], input_rows[0]["tokens"]))


Document: EventStoryLine - 1_10ecbplus - 1_10ecbplus
Source characters: 745
Sentences: 6
Source tokens: 147
Gold events (not exposed to pipeline): 15
Gold evaluated relations: 20
Ignored null pairs: 1
OWL-Time classes loaded: 23
OWL-Time properties loaded: 62
Controlled task relations: 2
Relation IDs: ['PRECONDITION', 'FALLING_ACTION']
Layer 2 workers: 16
Mention-free relation schemas injected: 0
Five-document input IDs: ['EventStoryLine - 1_10ecbplus', 'EventStoryLine - 1_11ecbplus', 'EventStoryLine - 1_12ecbplus', 'EventStoryLine - 1_13ecbplus', 'EventStoryLine - 1_14ecbplus']


### Indexed source table used by Layer 1

[S0] http : / / articles . latimes . com / 2013 / may / 03 / local / la - me - 0504 - lohan - rehab - 20130504
TOKENS 0=http 1=: 2=/ 3=/ 4=articles 5=. 6=latimes 7=. 8=com 9=/ 10=2013 11=/ 12=may 13=/ 14=03 15=/ 16=local 17=/ 18=la 19=- 20=me 21=- 22=0504 23=- 24=lohan 25=- 26=rehab 27=- 28=20130504

[S1] Lindsay Lohan checks into Betty Ford Center
TOKENS 0=Lindsay 1=Lohan 2=checks 3=into 4=Betty 5=Ford 6=Center

[S2] May 03 , 2013
TOKENS 0=May 1=03 2=, 3=2013

[S3] After skipping out on entering a Newport Beach rehabilitation facility and facing the prospect of arrest for violating her probation , Lindsay Lohan has checked into the Betty Ford Center to begin a 90 - day court - mandated stay in her reckless driving conviction .
TOKENS 0=After 1=skipping 2=out 3=on 4=entering 5=a 6=Newport 7=Beach 8=rehabilitation 9=facility 10=and 11=facing 12=the 13=prospect 14=of 15=arrest 16=for 17=violating 18=her 19=probation 20=, 21=Lindsay 22=Lohan 23=has 24=checked 25=into 26=the 27=Betty 28=Fo

## 3. Run the full native Layer 0--12 pipeline

In [4]:
if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        task_guidance_path=TASK_GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)

[NeoOLAF] Run directory: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_2_owltime_validated_events
[NeoOLAF] from_layer=0, to_layer=12, skip_layers=None
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction', 'layer02_candidate_enrichment', 'layer03_candidate_typing_resolution', 'layer04_candidate_relation_extraction', 'layer05_candidate_triple_generation', 'layer06_concept_relation_induction', 'layer07_hierarchisation', 'layer08_axiom_schemata_extraction', 'layer09_general_axiom_extraction', 'layer10_validation_reasoning', 'layer11_inference_completion', 'layer12_serialization']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.00s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_lingu

[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 0.01s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction
[NeoOLAF][Layer 4] strategy=structured_exact_then_native_parallel_fallback; parallel_workers=8; attempts=1
[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.00s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.00s
[NeoOLAF] Layer 6/12: layer06_concept_relation_induction

[NeoOLAF] Starting layer: layer06_concept_relation_induction
[NeoOLAF][Layer 6] deterministic ontology-aware concept induction for 6 node candidates; no LLM calls.
[NeoOLAF][Layer 6] deterministic ontology-aware relation induction for 5 relation candidates; no LLM calls.
[NeoOLAF] Finished layer: layer06_concept_relation_induction in 0.00s
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontology_aware_axiom_schema_generation
[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.00s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction
[NeoO

[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.00s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion
[NeoOLAF][Layer 11] strategy=ontology_aware_semantic_completion
[NeoOLAF][Layer 11] deterministic completion; max_concurrency=16; no LLM calls.
[NeoOLAF] Finished layer: layer11_inference_completion in 0.00s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_2_owltime_validated_events/exports
[NeoOLAF] Finished layer: layer12_serialization in 0.03s
[NeoOLAF] Pipeline finished in 8.95s
[NeoOLAF] Saved checkpoint: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_2_owltime_validated_events/checkpoints/after_selected_pipeline.pkl.gz
[Ne

### Runtime evidence saved

- `run_manifest.json` records the external OWL-Time path and loaded class/property counts.
- `run_logs/layer01_event_inventory.json`
- `run_logs/layer01_relation_generation.json`
- `run_logs/layer01_event_relation_instances.json`
- `run_logs/layer02_relation_decisions.json`
- `run_logs/layer02_compact_prompt_audit.json`
- `run_logs/layer04_endpoint_assignment.json`
- `run_logs/ontology_retrieval.jsonl`
- `layer_states/layer_00.json` through `layer_states/layer_12.json`


## 4. Strict event and relation evaluation

In [5]:
summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

display(pd.DataFrame(summary["layer_summary"]))

print("Strict relation evaluation; null relations excluded")
display(pd.DataFrame([summary["strict_relation_evaluation"]]))

print("Event mention inventory evaluation")
display(pd.DataFrame([summary["event_entity_evaluation"]]))

print("Relation-endpoint event inventory evaluation")
display(pd.DataFrame([summary["relation_endpoint_evaluation"]]))

print("Per-relation metrics")
display(pd.DataFrame(summary["per_relation_metrics"]))

print("Cumulative evaluation")
display(pd.DataFrame(summary["cumulative_evaluation"]))

print("First-failure counts")
print(summary["failure_counts"])

,layer,layer_name,linguistic_expressions,enriched_expressions,entity_candidates,relation_candidates,attribute_candidates,event_candidates,candidate_relation_assertions,candidate_triples,concept_candidates,ontology_relation_candidates,concept_hierarchy_links,relation_hierarchy_links,axiom_schema_candidates,general_axiom_candidates,completion_candidates,validation_issues,reasoning_inferred_triples
0,0,layer00_preprocessing,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,layer01_linguistic_expression_extraction,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,layer02_candidate_enrichment,11,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,layer03_candidate_typing_resolution,11,11,0,5,0,6,0,0,0,0,0,0,0,0,0,0,0
4,4,layer04_candidate_relation_extraction,11,11,0,5,0,6,5,0,0,0,0,0,0,0,0,0,0
5,5,layer05_candidate_triple_generation,11,11,0,5,0,6,5,5,0,0,0,0,0,0,0,0,0
6,6,layer06_concept_relation_induction,11,11,0,5,0,6,5,5,0,2,0,0,0,0,0,0,0
7,7,layer07_hierarchisation,11,11,0,5,0,6,5,5,0,2,0,2,0,0,0,0,0
8,8,layer08_axiom_schemata_extraction,11,11,0,5,0,6,5,5,0,2,0,2,6,0,0,0,0
9,9,layer09_general_axiom_extraction,11,11,0,5,0,6,5,5,0,2,0,2,6,8,0,0,0


Strict relation evaluation; null relations excluded


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,4,20,0,4,20,0.0,0.0,0.0


Event mention inventory evaluation


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,5,15,5,0,10,1.0,0.333333,0.5


...

Per-relation metrics


,relation_id,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,PRECONDITION,2,7,0,2,7,0.0,0.0,0.0
1,FALLING_ACTION,2,13,0,2,13,0.0,0.0,0.0


Cumulative evaluation


,layer,layer_name,relation_predicted,relation_gold,relation_true_positive,relation_false_positive,relation_false_negative,relation_precision,relation_recall,relation_f1,event_predicted,event_gold,event_true_positive,event_false_positive,event_false_negative,event_precision,event_recall,event_f1
0,0,layer00_preprocessing,0,20,0,0,20,0.0,0.0,0.0,0,15,0,0,15,0.0,0.000000,0.0
1,1,layer01_linguistic_expression_extraction,0,20,0,0,20,0.0,0.0,0.0,5,15,5,0,10,1.0,0.333333,0.5
2,2,layer02_candidate_enrichment,0,20,0,0,20,0.0,0.0,0.0,5,15,5,0,10,1.0,0.333333,0.5
3,3,layer03_candidate_typing_resolution,0,20,0,0,20,0.0,0.0,0.0,5,15,5,0,10,1.0,0.333333,0.5
4,4,layer04_candidate_relation_extraction,4,20,0,4,20,0.0,0.0,0.0,5,15,5,0,10,1.0,0.333333,0.5
5,5,layer05_candidate_triple_generation,4,20,0,4,20,0.0,0.0,0.0,5,15,5,0,10,1.0,0.333333,0.5
6,6,layer06_concept_relation_induction,4,20,0,4,20,0.0,0.0,0.0,5,15,5,0,10,1.0,0.333333,0.5
7,7,layer07_hierarchisation,4,20,0,4,20,0.0,0.0,0.0,5,15,5,0,10,1.0,0.333333,0.5
8,8,layer08_axiom_schemata_extraction,4,20,0,4,20,0.0,0.0,0.0,5,15,5,0,10,1.0,0.333333,0.5
9,9,layer09_general_axiom_extraction,4,20,0,4,20,0.0,0.0,0.0,5,15,5,0,10,1.0,0.333333,0.5


First-failure counts
{'layer01_source_event_missing': 12, 'layer01_target_event_missing': 8}


## 5. Layer 1 validated event inventory and closed-inventory relation instances

In [6]:
layer1_rows = read_json(RUN_DIR / "run_logs/layer01_event_relation_instances.json")
layer1_df = pd.DataFrame(layer1_rows)
display(layer1_df)

accepted = layer1_df[layer1_df["status"] == "accepted"] if not layer1_df.empty else layer1_df
if not accepted.empty:
    print("Accepted event mentions:", int((accepted["label"] == "event_mention").sum()))
    print("Accepted relation instances:", int((accepted["label"] == "relation_instance").sum()))
    print("Rejected rows:", int((layer1_df["status"] == "rejected").sum()))

,phase,status,expr_id,text,label,relation_instance,justification
0,materialization,accepted,expr_00000,S1[2:4]::checks into,event_mention,None,Explicit admission to a facility.
1,materialization,accepted,expr_00001,S3[24:26]::checked into,event_mention,None,Explicit admission to a facility.
2,materialization,accepted,expr_00002,S3[31:32]::begin,event_mention,None,Initiation of a court‑mandated stay.
3,materialization,accepted,expr_00003,S5[9:12]::rear - ended,event_mention,None,Vehicle collision event.
4,materialization,accepted,expr_00004,S5[27:28]::lied,event_mention,None,Deception toward police.
5,materialization,accepted,expr_00005,S5[31:32]::telling,event_mention,None,Narrative of lying to police.
6,materialization,accepted,expr_00006,S3[24:26]::checked into || enables || S3[31:32...,relation_instance,"[S3[24:26]::checked into, enables, S3[31:32]::...",Checking into the Betty Ford Center establishe...
7,materialization,accepted,expr_00007,S5[9:12]::rear - ended || followed by || S5[27...,relation_instance,"[S5[9:12]::rear - ended, followed by, S5[27:28...",The rear‑ended collision was followed by Lohan...
8,materialization,accepted,expr_00008,S5[9:12]::rear - ended || followed by || S5[31...,relation_instance,"[S5[9:12]::rear - ended, followed by, S5[31:32...",The rear‑ended collision was followed by Lohan...
9,materialization,accepted,expr_00009,S5[9:12]::rear - ended || leads to || S3[24:26...,relation_instance,"[S5[9:12]::rear - ended, leads to, S3[24:26]::...",The collision and subsequent lie prompted Loha...


Accepted event mentions: 6
Accepted relation instances: 5
Rejected rows: 0


### Layer 1A span validation/repair and Layer 1B closed-inventory audit


In [7]:
inventory_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_event_inventory.json"))
relation_generation = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_relation_generation.json"))

print("Layer 1A event inventory audit")
display(inventory_audit)
if not inventory_audit.empty:
    print("Accepted validated events:", int((inventory_audit["status"] == "accepted").sum()))
    print("Rejected event proposals:", int((inventory_audit["status"] == "rejected").sum()))
    if "repaired" in inventory_audit:
        print("Source-token span repairs:", int(inventory_audit["repaired"].fillna(False).sum()))

print("Layer 1B relation generation audit")
display(relation_generation)
if not relation_generation.empty:
    print("Accepted closed-inventory pairs:", int((relation_generation["status"] == "accepted").sum()))
    print("Rejected pair proposals:", int((relation_generation["status"] == "rejected").sum()))


Layer 1A event inventory audit


,raw_item,proposed_event_key,proposed_sentence_id,proposed_token_start,proposed_token_end,proposed_trigger,repair_steps,status,reason,canonical_event_key,sentence_id,token_start,token_end,repaired_trigger,repaired,phase,justification
0,"{'sentence_id': 1, 'token_start': 2, 'token_en...",None,1,2,4,checks into,[],accepted,validated_source_token_span,S1[2:4]::checks into,1,2,4,checks into,False,event_inventory,Explicit admission to a facility.
1,"{'sentence_id': 3, 'token_start': 24, 'token_e...",None,3,24,26,checked into,[],accepted,validated_source_token_span,S3[24:26]::checked into,3,24,26,checked into,False,event_inventory,Explicit admission to a facility.
2,"{'sentence_id': 3, 'token_start': 31, 'token_e...",None,3,31,32,begin,[],accepted,validated_source_token_span,S3[31:32]::begin,3,31,32,begin,False,event_inventory,Initiation of a court‑mandated stay.
3,"{'sentence_id': 5, 'token_start': 9, 'token_en...",None,5,9,12,rear - ended,[],accepted,validated_source_token_span,S5[9:12]::rear - ended,5,9,12,rear - ended,False,event_inventory,Vehicle collision event.
4,"{'sentence_id': 5, 'token_start': 27, 'token_e...",None,5,27,28,lied,[],accepted,validated_source_token_span,S5[27:28]::lied,5,27,28,lied,False,event_inventory,Deception toward police.
5,"{'sentence_id': 5, 'token_start': 31, 'token_e...",None,5,31,32,telling,[],accepted,validated_source_token_span,S5[31:32]::telling,5,31,32,telling,False,event_inventory,Narrative of lying to police.


Accepted validated events: 6
Rejected event proposals: 0
Source-token span repairs: 0
Layer 1B relation generation audit


,phase,source_event_id,target_event_id,relation_cue,justification,status,reason,relation_instance
0,relation_inventory,E0001,E0002,enables,Checking into the Betty Ford Center establishe...,accepted,closed_inventory_relation,S3[24:26]::checked into || enables || S3[31:32...
1,relation_inventory,E0003,E0004,followed by,The rear‑ended collision was followed by Lohan...,accepted,closed_inventory_relation,S5[9:12]::rear - ended || followed by || S5[27...
2,relation_inventory,E0003,E0005,followed by,The rear‑ended collision was followed by Lohan...,accepted,closed_inventory_relation,S5[9:12]::rear - ended || followed by || S5[31...
3,relation_inventory,E0003,E0001,leads to,The collision and subsequent lie prompted Loha...,accepted,closed_inventory_relation,S5[9:12]::rear - ended || leads to || S3[24:26...
4,relation_inventory,E0004,E0005,followed by,Lohan lying to police was followed by her tell...,accepted,closed_inventory_relation,S5[27:28]::lied || followed by || S5[31:32]::t...


Accepted closed-inventory pairs: 5
Rejected pair proposals: 0


## 6. Parallel Layer 2 ontology decisions

In [8]:
layer2 = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_relation_decisions.json"))
prompt_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_compact_prompt_audit.json"))

display(layer2)
print("Compact prompt audit")
display(prompt_audit)
if not prompt_audit.empty:
    print("Layer 2 calls:", len(prompt_audit))
    print("Mean Layer 2 user characters:", round(prompt_audit["user_chars"].mean(), 1))
    print("Maximum Layer 2 user characters:", int(prompt_audit["user_chars"].max()))
    print("Candidates per decision:", sorted(set(len(x) for x in prompt_audit["candidate_relation_ids"])))

,expr_id,relation_instance,source,predicate,target,found,selected_relation_id,canonical_relation,decision,ontology_hints
0,expr_00006,S3[24:26]::checked into || enables || S3[31:32...,S3[24:26]::checked into,enables,S3[31:32]::begin,True,PRECONDITION,PRECONDITION,The act of checking into the Betty Ford Center...,"[controlled_relation:PRECONDITION, promote_to_..."
1,expr_00007,S5[9:12]::rear - ended || followed by || S5[27...,S5[9:12]::rear - ended,followed by,S5[27:28]::lied,True,FALLING_ACTION,FALLING_ACTION,The rear‑ended collision is a preceding event ...,"[controlled_relation:FALLING_ACTION, promote_t..."
2,expr_00008,S5[9:12]::rear - ended || followed by || S5[31...,S5[9:12]::rear - ended,followed by,S5[31:32]::telling,True,FALLING_ACTION,FALLING_ACTION,The telling of police is a downstream conseque...,"[controlled_relation:FALLING_ACTION, promote_t..."
3,expr_00009,S5[9:12]::rear - ended || leads to || S3[24:26...,S5[9:12]::rear - ended,leads to,S3[24:26]::checked into,True,PRECONDITION,PRECONDITION,The collision (rear ended) serves as a prerequ...,"[controlled_relation:PRECONDITION, promote_to_..."
4,expr_00010,S5[27:28]::lied || followed by || S5[31:32]::t...,S5[27:28]::lied,followed by,S5[31:32]::telling,True,PRECONDITION,PRECONDITION,The act of lying to police establishes the con...,"[controlled_relation:PRECONDITION, promote_to_..."


Compact prompt audit


,expr_id,relation_instance,candidate_relation_ids,system_chars,user_chars
0,expr_00006,S3[24:26]::checked into || enables || S3[31:32...,"[PRECONDITION, FALLING_ACTION]",714,4770
1,expr_00007,S5[9:12]::rear - ended || followed by || S5[27...,"[PRECONDITION, FALLING_ACTION]",714,3952
2,expr_00008,S5[9:12]::rear - ended || followed by || S5[31...,"[PRECONDITION, FALLING_ACTION]",714,3974
3,expr_00009,S5[9:12]::rear - ended || leads to || S3[24:26...,"[PRECONDITION, FALLING_ACTION]",714,4116
4,expr_00010,S5[27:28]::lied || followed by || S5[31:32]::t...,"[PRECONDITION, FALLING_ACTION]",714,3962


Layer 2 calls: 5
Mean Layer 2 user characters: 4154.8
Maximum Layer 2 user characters: 4770
Candidates per decision: [2]


## 7. Strict gold-relation trace

In [9]:
trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace.csv")
display(trace_df)
display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)

,source_event_id,relation_id,target_event_id,source_keys,target_keys,first_failure
0,EVENT_2ea9a901cc230afcc371bcffee02a551,FALLING_ACTION,EVENT_401ce8fe35db27ca7b5f7cf83acc4084,"[""S3[39:40]::stay""]","[""S5[9:12]::rear - ended""]",layer01_source_event_missing
1,EVENT_2ea9a901cc230afcc371bcffee02a551,FALLING_ACTION,EVENT_7d29e127b36f34a94ceb0f13f9ea3a3b,"[""S3[39:40]::stay""]","[""S3[44:45]::conviction""]",layer01_source_event_missing
2,EVENT_2ea9a901cc230afcc371bcffee02a551,FALLING_ACTION,EVENT_a8934f381d6d835057ba8c537386ff9e,"[""S3[39:40]::stay""]","[""S5[27:28]::lied""]",layer01_source_event_missing
3,EVENT_2ea9a901cc230afcc371bcffee02a551,FALLING_ACTION,EVENT_f83b48c549abe22d26dc21624d8de397,"[""S3[39:40]::stay""]","[""S5[31:32]::telling""]",layer01_source_event_missing
4,EVENT_34cccabacad4dea93d3e762114dc05cd,FALLING_ACTION,EVENT_b79df11f8737f700691af4f8a7132190,"[""S1[2:4]::checks into""]","[""S3[4:5]::entering""]",layer01_target_event_missing
5,EVENT_34cccabacad4dea93d3e762114dc05cd,FALLING_ACTION,EVENT_c332a6b60af0495f34856f112dc68632,"[""S1[2:4]::checks into""]","[""S3[15:16]::arrest""]",layer01_target_event_missing
6,EVENT_34cccabacad4dea93d3e762114dc05cd,PRECONDITION,EVENT_2ea9a901cc230afcc371bcffee02a551,"[""S1[2:4]::checks into""]","[""S3[39:40]::stay""]",layer01_target_event_missing
7,EVENT_34cccabacad4dea93d3e762114dc05cd,PRECONDITION,EVENT_a6c700ffee90bb400461d1687e2287b9,"[""S1[2:4]::checks into""]","[""S5[2:3]::stay""]",layer01_target_event_missing
8,EVENT_434faad0c9ec71baa6a91df3c385299b,FALLING_ACTION,EVENT_70540222392ed30d611c5073a5b3204c,"[""S4[15:16]::violation""]","[""S4[21:22]::case""]",layer01_source_event_missing
9,EVENT_7d29e127b36f34a94ceb0f13f9ea3a3b,PRECONDITION,EVENT_a6c700ffee90bb400461d1687e2287b9,"[""S3[44:45]::conviction""]","[""S5[2:3]::stay""]",layer01_source_event_missing


,first_failure,gold_relations
0,layer01_source_event_missing,12
1,layer01_target_event_missing,8


## 8. Native event candidates, assertions and triples

In [10]:
states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer11 = states.get(11)

if layer3:
    print("Layer 3 event candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "ontology_hints": c.ontology_hints,
    } for c in layer3.event_candidates or []]))

    print("Layer 3 relation candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "controlled_hints": [h for h in c.ontology_hints if str(h).lower().startswith("controlled_relation:")],
    } for c in layer3.relation_candidates or []]))
    assert all(c.mentions for c in layer3.relation_candidates or []), "Mention-free relation candidate detected."

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 triples")
    display(pd.DataFrame([{
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Layer 6 ontology relation candidates:", len(layer6.ontology_relation_candidates or []) if layer6 else None)
print("Layer 11 completion candidates:", len(layer11.completion_candidates or []) if layer11 else None)

Layer 3 event candidates


,candidate_id,canonical_label,mentions,ontology_hints
0,cand_s_00000,S1[2:4]::checks into,[S1[2:4]::checks into],"[semantic_role:event_mention, candidate_family..."
1,cand_s_00001,S3[24:26]::checked into,[S3[24:26]::checked into],"[semantic_role:event_mention, candidate_family..."
2,cand_s_00002,S3[31:32]::begin,[S3[31:32]::begin],"[semantic_role:event_mention, candidate_family..."
3,cand_s_00003,S5[9:12]::rear - ended,[S5[9:12]::rear - ended],"[semantic_role:event_mention, candidate_family..."
4,cand_s_00004,S5[27:28]::lied,[S5[27:28]::lied],"[semantic_role:event_mention, candidate_family..."
5,cand_s_00005,S5[31:32]::telling,[S5[31:32]::telling],"[semantic_role:event_mention, candidate_family..."


Layer 3 relation candidates


,candidate_id,canonical_label,mentions,controlled_hints
0,cand_r_00000,PRECONDITION,[S3[24:26]::checked into || enables || S3[31:3...,[controlled_relation:PRECONDITION]
1,cand_r_00001,FALLING_ACTION,[S5[9:12]::rear - ended || followed by || S5[2...,[controlled_relation:FALLING_ACTION]
2,cand_r_00002,FALLING_ACTION,[S5[9:12]::rear - ended || followed by || S5[3...,[controlled_relation:FALLING_ACTION]
3,cand_r_00003,PRECONDITION,[S5[9:12]::rear - ended || leads to || S3[24:2...,[controlled_relation:PRECONDITION]
4,cand_r_00004,PRECONDITION,[S5[27:28]::lied || followed by || S5[31:32]::...,[controlled_relation:PRECONDITION]


Layer 4 assertions


,source,predicate,target,confidence
0,S3[24:26]::checked into,PRECONDITION,S3[31:32]::begin,1.0
1,S5[9:12]::rear - ended,FALLING_ACTION,S5[27:28]::lied,1.0
2,S5[9:12]::rear - ended,FALLING_ACTION,S5[31:32]::telling,1.0
3,S5[9:12]::rear - ended,PRECONDITION,S3[24:26]::checked into,1.0
4,S5[27:28]::lied,PRECONDITION,S5[31:32]::telling,1.0


Layer 5 triples


,subject,predicate,object,confidence
0,S3[24:26]::checked into,PRECONDITION,S3[31:32]::begin,1.0
1,S5[9:12]::rear - ended,FALLING_ACTION,S5[27:28]::lied,1.0
2,S5[9:12]::rear - ended,FALLING_ACTION,S5[31:32]::telling,1.0
3,S5[9:12]::rear - ended,PRECONDITION,S3[24:26]::checked into,1.0
4,S5[27:28]::lied,PRECONDITION,S5[31:32]::telling,1.0


Layer 6 ontology relation candidates: 2
Layer 11 completion candidates: 0


## 9. Event projection audit

In [11]:
projection = pd.read_csv(RUN_DIR / "analysis/event_projection_audit.csv")
display(projection)

# Exact-span demonstration using source structure. Gold is consulted only here,
# after the pipeline has completed.
first_gold_key = next(iter(gold_event_index(gold)["keys_by_id"].values()))[0]
print(first_gold_key)
print(project_event_label(first_gold_key, gold))

,event_id,method,label,candidate_event_ids
0,EVENT_34cccabacad4dea93d3e762114dc05cd,exact_sentence_token_span,S1[2:4]::checks into,NaN
1,EVENT_d7aec1e4ff20c3264f9fb7686355eb8a,exact_sentence_token_span,S3[24:26]::checked into,NaN
2,NaN,unmapped_or_ambiguous,S3[31:32]::begin,[]
3,EVENT_401ce8fe35db27ca7b5f7cf83acc4084,exact_sentence_token_span,S5[9:12]::rear - ended,NaN
4,EVENT_a8934f381d6d835057ba8c537386ff9e,exact_sentence_token_span,S5[27:28]::lied,NaN
5,EVENT_f83b48c549abe22d26dc21624d8de397,exact_sentence_token_span,S5[31:32]::telling,NaN


S4[15:16]::violation
{'event_id': 'EVENT_434faad0c9ec71baa6a91df3c385299b', 'method': 'exact_sentence_token_span', 'label': 'S4[15:16]::violation'}


## 10. Speed, concurrency and errors

In [12]:
def optional_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(path)) if path.is_file() else pd.DataFrame()

calls = optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
retrieval = optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_system_chars=("system_chars", "mean"),
        mean_user_chars=("user_chars", "mean"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("Backend/API errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)
if not retrieval.empty:
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))

manifest = read_json(RUN_DIR / "run_manifest.json")
print("Wall-clock pipeline seconds:", manifest.get("elapsed_seconds"))

,call_index,layer_tag,model,temperature,message_count,system_chars,user_chars,max_tokens,request_timeout,started_at,status,elapsed_seconds,response_chars,response_path,json_parse_ok,parsed_type,parse_error
0,1,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1431,5984,8192,180,2026-07-30 21:02:01,ok,4.812,1610,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
1,2,layer01_event_instances,openai/gpt-oss-20b,0.0,2,1280,6043,8192,180,2026-07-30 21:02:06,ok,1.661,1224,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
2,7,layer02_event_relation_linking,openai/gpt-oss-20b,0.0,2,714,3962,384,60,2026-07-30 21:02:08,ok,1.030,254,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
3,3,layer02_event_relation_linking,openai/gpt-oss-20b,0.0,2,714,4770,384,60,2026-07-30 21:02:08,ok,1.454,231,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
4,6,layer02_event_relation_linking,openai/gpt-oss-20b,0.0,2,714,4116,384,60,2026-07-30 21:02:08,ok,1.856,239,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
5,4,layer02_event_relation_linking,openai/gpt-oss-20b,0.0,2,714,3952,384,60,2026-07-30 21:02:08,ok,1.990,215,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
6,5,layer02_event_relation_linking,openai/gpt-oss-20b,0.0,2,714,3974,384,60,2026-07-30 21:02:08,ok,2.295,183,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None


,layer_tag,calls,total_recorded_seconds,maximum_call_seconds,mean_system_chars,mean_user_chars,mean_response_chars
0,layer01_event_instances,2,6.473,4.812,1355.5,6013.5,1417.0
1,layer02_event_relation_linking,5,8.625,2.295,714.0,4154.8,224.4


Backend/API errors: 0
JSON parse errors: 0


,layer_name,retrieval_calls
0,layer02_candidate_enrichment,5


Wall-clock pipeline seconds: 8.955


## Success checklist before the five-document batch

1. The preflight resolves the sibling RAGTree OWL-Time file.
2. Layer 1A emits validated/repaired indexed event keys; Layer 1B uses only that closed inventory for directed relation instances.
3. Layer 2 maps only to `PRECONDITION`, `FALLING_ACTION`, or `found=false`.
4. Layer 4 resolves exact event endpoints without direction swaps.
5. Analysis completes without the former `LAYER_NAMES.get` error.
6. Gold remains unavailable until post-pipeline evaluation.
